[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-12-final-revision.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 12 — Final revision

**MSIS · AME 5053 · Week 12 · 3 hours**

The dataset contains fictional municipal utility reports with gold sentiment labels; the search queries, maintenance circular and QA questions are defined in the data-loading cell. Complete the assessed programming work in the empty code cells below. You may add further cells and explanatory Markdown

This notebook is designed for **Google Colab with internet access** (the models download from Hugging Face). Run the supplied environment and data-loading cells in order. Do not alter the supplied dataset, gold labels, model identifiers, or the model-loading code. Write the assessed programs independently. Save the completed notebook with all outputs visible and submit it as instructed by the examiner.



**Instructions:** Use the supplied dataset (https://learnermanipal-my.sharepoint.com/:x:/g/personal/shripada_hebbar_manipal_edu/IQCOdBIEt3YqQbVS9b_vPXj5AdbQT8WqTymxCGPS6t7kSHE?e=gehfWD) and notebook. The environment, data-loading, and pretrained-model setup are provided. Develop the assessed programs independently. Preserve the supplied dataset and labels. Submit the executed notebook with the required outputs. The notebook is designed for Google Colab with internet access; the two search queries, the maintenance circular and the four questions are defined in the supplied notebook.

**1.** A municipal water and electricity utility receives reports from residents about supply, repairs, meters and billing. The supplied dataset contains 18 reports with request references, dates and sentiment labels. Develop a Python program to perform the following tasks:

- **(a)** Extract request references, dates and report bodies using regular expressions. Normalise the report bodies by lowercasing, removing punctuation and digits, and removing scikit-learn's English stop words (no stemming or lemmatisation); keep the request reference, date, original body and processed body as table columns. Using an inverted index and set operations, implement Boolean retrieval for `water AND (leak OR leakage)` case-insensitively. Display the extracted table and matching request references.
- **(b)** Build a TF-IDF representation of the processed report bodies using unigram and bigram features. Process each of the two supplied search queries in the same way, calculate cosine similarity and display the five highest-ranked requests with their scores. Display the number of distinct features in the fitted representation and briefly explain how query terms absent from the vocabulary affect the ranking.

**2.** The utility also intends to classify resident reports and provide factual answers from its maintenance circular. Use the provided pretrained BERT-family models, resident reports, gold-standard labels and maintenance circular to develop the following applications:

- **(a)** Construct a sentiment classification workflow for the report bodies extracted in Question 1(a) (if extraction is incomplete, use the text after "Report:"). Display the predicted class and both class probabilities for each record. Compute the confusion matrix, accuracy and macro-F1 against the supplied labels, and compare the model with a majority-class baseline that predicts NEGATIVE for every record (the classes are balanced, so NEGATIVE is fixed as the tie-break). Produce a table of incorrectly classified records, stating explicitly if there are none, and apply the model to an original minimal pair of sentences differing in one meaningful expression; report the change in positive-class probability.
- **(b)** Develop an extractive question-answering program for the four supplied questions and the maintenance circular. Return the model's answer span and score for each question. Check each answer against the maintenance circular: an answer is supported only if the span correctly answers the question. Produce a final table with the question, model span, score, supported (Yes/No), final answer.



### A. Environment and dataset

In [ ]:
%pip -q install "transformers>=4.55,<5" "scikit-learn>=1.5,<2" "pandas>=2,<3"

In [ ]:
import io
import re
import numpy as np
import pandas as pd
import torch
try:
    from google.colab import files  # Colab upload widget
except ImportError:
    files = None  # local Jupyter: place the CSV next to this notebook
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

np.random.seed(42)
torch.manual_seed(42)
print('Libraries ready; PyTorch:', torch.__version__)


In [ ]:
DATA_FILENAME = 'data.csv'
if files is not None:
    print(f'Upload {DATA_FILENAME} when prompted.')
    uploaded = files.upload()
    # Colab renames repeat uploads to e.g. 'data (1).csv'
    matches = [k for k in uploaded if k.startswith(DATA_FILENAME[:-4])]
    assert matches, f'Expected {DATA_FILENAME}; got {list(uploaded)}'
    feedback = pd.read_csv(io.BytesIO(uploaded[matches[-1]]), encoding='utf-8').to_dict(orient='records')
else:
    feedback = pd.read_csv(DATA_FILENAME, encoding='utf-8').to_dict(orient='records')
assert len(feedback) == 18 and all('message' in row for row in feedback)
queries = ['electricity supply restoration', 'monthly water charges']
notice = 'Lakeview Municipal Utilities Maintenance Circular, 26 October 2026. Water supply to Ward 12 and Ward 14 will be suspended on 9 November 2026 for pipeline replacement. Supply is expected to resume by 18:00 on the same day. Tanker water will be provided at the Ward 12 community hall from 08:00. Electricity dues can be paid at the Civic Centre counter between 09:00 and 17:00 on working days. Leakage complaints can be registered by calling 155-300 at any time. Residents should store sufficient drinking water before the shutdown.'
qa_questions = ['On which date will water supply to Ward 12 be suspended?', 'By what time is the water supply expected to resume?', 'Where will tanker water be provided?', 'How much compensation will residents receive for the shutdown?']

print('Resident reports:', len(feedback), '| Retrieval queries:', len(queries),
      '| QA questions:', len(qa_questions))
display(pd.DataFrame(feedback).head(3))

### B. Provided BERT-family model setup

In [ ]:
# Provided setup only: the assessed prediction and evaluation code is NOT supplied.
SENTIMENT_CHECKPOINT = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
QA_CHECKPOINT = 'distilbert/distilbert-base-cased-distilled-squad'
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_CHECKPOINT)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_CHECKPOINT)
sentiment_model.to('cpu').eval()
qa_model = pipeline('question-answering', model=QA_CHECKPOINT, device=-1)
print('Both pretrained BERT-family task models are ready on CPU.')


## Question 1 - Text processing and vector-based retrieval

Use the supplied resident reports and the two queries in `queries`. Implement the two parts described in the question paper.

- **1(a):** extracted table (request reference, date, original body, processed body) and the request references matching `water AND (leak OR leakage)`, using an inverted index.
- **1(b):** the top-5 ranked list with cosine scores for each query, the number of TF-IDF features, and a brief explanation of how query terms absent from the vocabulary affect the ranking.

## Question 2 - BERT applications and evaluation

Use the supplied pretrained models, the report bodies extracted in Question 1(a) (if extraction is incomplete, use the text after `Report:`), the gold labels, the maintenance circular in `notice` and the questions in `qa_questions`.

- **2(a):** predicted class and both class probabilities per record; confusion matrix, accuracy and macro-F1; comparison with an all-NEGATIVE majority baseline; table of misclassified records (state explicitly if none); original minimal pair and change in positive-class probability.
- **2(b):** span and score per question; final table with question, span, score, supported (Yes/No), final answer and one-line justification, using `NOT STATED IN SOURCE` for unsupported requests.

## Submission check

Ensure that all assessed cells have been executed, the result tables are visible, and the explanation for 1(b) and the justifications for 2(b) are present. You can share the link to your final jupyter notebook via email.